<a href="https://colab.research.google.com/github/wenxy59/program26summer/blob/main/notebooks/MiniBioDesignBench_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniBioDesignBench Colab Walkthrough

Main Colab link: [Open MiniBioDesignBench from GitHub](https://colab.research.google.com/github/wenxy59/program26summer/blob/main/notebooks/MiniBioDesignBench_Colab.ipynb)

This GitHub-Colab link loads the notebook directly from the `main` branch of the public repository. When the repository is updated, reopen the link to get the newest notebook. If you save a Google Drive copy later, treat that copy as your personal workspace.

This notebook lets you run MiniBioDesignBench without logging into a remote server and without installing Python on your own computer.

**What you should do first**

1. Read the short explanations before each code cell.
2. Click **Runtime -> Run all** for the first run.
3. After it finishes, come back and inspect the output one section at a time.

The default demo does **not** need GPU, CUDA, conda, OpenAI keys, or server access.


## 0. Setup: clone the project first

Run this cell before every other code cell.

Colab starts from a temporary Linux folder. This cell downloads the project into `/content/program26summer` and moves the notebook into that folder, so all later commands can find `run_demo.py`, `data/`, and `src/`.

It is safe to rerun this cell. It first moves back to `/content`, removes the old project copy, then clones a fresh copy from GitHub.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/wenxy59/program26summer.git"
PROJECT_DIR = Path("/content/program26summer")

%cd /content
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"

print("Now working in:")
print(Path.cwd())


## 1. What is Google Colab?

Google Colab is a Jupyter notebook that runs in your browser.

The important idea is that the notebook has two parts:

- The page you see in the browser: text, code cells, and output.
- A temporary Linux computer in the cloud: this is called the **runtime**.

When you run a code cell, the code is executed on that temporary Linux computer, not on your laptop.

Useful Colab vocabulary:

- **Runtime**: the temporary cloud computer attached to this notebook.
- **Cell**: one block of text or code.
- **Markdown cell**: explanation text like this one.
- **Code cell**: a block you can run.
- `command`: run a Linux command, for example `ls` or `python3 run_demo.py`.
- `cd path`: change the working folder for later cells.

The runtime can disconnect or reset. If that happens, rerun the setup cells from the top.

## 2. Project path rules

Colab starts in:

```text
/content
```

This notebook will clone the GitHub repository into:

```text
/content/program26summer
```

After cloning, all project commands should run from:

```text
/content/program26summer
```

Inside that folder, the important files are:

```text
README.md
run_demo.py
data/tasks.json
data/gold_submissions.json
src/grader.py
src/agents.py
src/pareto.py
```

If you see an error like `No such file or directory`, rerun Section 0 first and confirm that the notebook moved into `/content/program26summer`.

## 3. Check the Colab runtime

This cell prints basic information about the cloud computer Colab gave you.

Because Section 0 already cloned the project, this cell should print `/content/program26summer` as the current directory.


In [ ]:
import os
import platform
import sys
from pathlib import Path

print("Python version:")
print(sys.version)
print()
print("Platform:")
print(platform.platform())
print()
print("Current directory:")
print(Path.cwd())

## 4. Optional GPU check

MiniBioDesignBench is small and deterministic, so CPU is enough.

If the next cell says `No GPU found`, that is completely fine.

In [ ]:
!nvidia-smi || echo "No GPU found. This is OK because the demo only needs CPU."

## 5. Tiny Linux warm-up inside the project folder

Colab is a Linux environment. These commands are useful:

- `pwd`: print current folder.
- `ls`: list files in the current folder.
- `python3 --version`: check Python.

The exclamation mark `!` tells Colab to run a Linux command. Because we already cloned the repository, `pwd` should print `/content/program26summer`, and `ls` should show files such as `README.md`, `run_demo.py`, `data`, and `src`.


In [ ]:
!pwd
!ls -lah
!python3 --version

## 6. Check the project files

Before running the benchmark, confirm that the expected files exist.

This is a good habit: many beginner errors are just path errors.

In [ ]:
from pathlib import Path

expected_files = [
    "README.md",
    "run_demo.py",
    "data/tasks.json",
    "data/gold_submissions.json",
    "src/grader.py",
    "src/agents.py",
    "src/pareto.py",
]

for file_name in expected_files:
    path = Path(file_name)
    print(("FOUND " if path.exists() else "MISSING"), file_name)

print("\nProject tree:")
!find . -maxdepth 2 -type f | sort

## 7. Run the required smoke test

The file `data/gold_submissions.json` contains known-valid answers.

The grader checks whether those answers satisfy the hard constraints in `data/tasks.json`.

If this cell passes, the dataset and grader are working.

In [ ]:
!python3 run_demo.py --mode grade-gold

Expected final line:

```text
Gold submissions: 8/8 pass
```

If you get a different result, do not edit randomly. Read the printed failure reason first.

## 8. Run all deterministic demos

This command runs three parts:

1. gold-solution grading;
2. deterministic agent baselines;
3. Pareto-front reports for multi-objective candidate tasks.

The baseline agents are not ChatGPT calls. They are simple local functions, so this part is reproducible.

In [ ]:
!python3 run_demo.py --mode all

## 9. Inspect the dataset

Now we read the task JSON files directly.

Each task has:

- `task_id`: unique task name;
- `prompt`: natural-language instruction;
- `inputs`: structured data for the task;
- `hard_constraints`: rules that must pass;
- `soft_objectives`: goals used for trade-off analysis.

In [ ]:
import json
from pathlib import Path

tasks = json.loads(Path("data/tasks.json").read_text())
gold = json.loads(Path("data/gold_submissions.json").read_text())

print("Number of tasks:", len(tasks))
print()
for i, task in enumerate(tasks, 1):
    print(f"{i:02d}. {task['task_id']}")
    print(f"    level:    {task.get('level')}")
    print(f"    category: {task.get('category')}")
    print(f"    prompt:   {task.get('prompt')[:100]}...")

## 10. Inspect one task in detail

Change `TASK_INDEX` to inspect a different task.

Python uses zero-based indexing:

- `TASK_INDEX = 0` means the first task.
- `TASK_INDEX = 1` means the second task.

In [ ]:
TASK_INDEX = 0

task = tasks[TASK_INDEX]
task_id = task["task_id"]

print("TASK")
print(json.dumps(task, indent=2))
print("\nGOLD SUBMISSION")
print(json.dumps(gold[task_id], indent=2))

## 11. Understand what the grader reports

The grader does not only say pass/fail. It also explains which constraint failed.

The next cell intentionally makes a bad answer for the first task so you can see the feedback format.

In [ ]:
from src.grader import grade

task = tasks[0]
bad_submission = {
    "sequence": "ATGAAA",
    "rationale": "This is intentionally too short and should fail."
}

result = grade(task, bad_submission)
print(json.dumps(result, indent=2))

## 12. Exercise: failure analysis

Pick one task and answer these questions in your notes:

1. What does the prompt ask for?
2. Which hard constraints must always pass?
3. Which soft objectives create trade-offs?
4. What would a naive one-shot answer likely get wrong?
5. How could a repair agent use grader feedback to improve the answer?

## 13. Optional: edit tasks in Colab

You can edit files from the left sidebar:

1. Click the folder icon on the left.
2. Open `program26summer/data/tasks.json`.
3. Open `program26summer/data/gold_submissions.json`.
4. Make small edits.
5. Rerun the smoke test.

Important: Colab is temporary. If you close the runtime without saving, your edits may disappear.

## 14. Save your work

For beginners, the safest method is to download a zip file.

If you later learn Git, you can also commit to a branch and open a pull request.

In [ ]:
try:
    from google.colab import files
    !zip -r minibio_colab_work.zip README.md COLAB_GUIDE.md run_demo.py data src requirements.txt LICENSE notebooks
    files.download("minibio_colab_work.zip")
except Exception as exc:
    print("This download helper only works inside Google Colab.")
    print("Error:", exc)

## 15. Troubleshooting

Common problems:

- **`No such file or directory` during clone**: the runtime may still be inside a deleted project folder. Run `%cd /content`, then rerun the clone cell.
- **Runtime disconnected**: reconnect and run from the top.
- **JSON error**: check commas, brackets, and quotation marks in `tasks.json` or `gold_submissions.json`.
- **Gold test fails after editing**: read the failure reason; then fix either the task constraints or the gold submission.

When in doubt, rerun from Section 0.